# DBSCAN with Scikit-Learn

### Set up and generate data

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
%matplotlib inline

In [ ]:
X, label = make_moons(n_samples=200, noise=0.1, random_state=19)
print X[:5,]

### Train the model

In [ ]:
# model = DBSCAN?
model = DBSCAN(eps=0.25, min_samples=12).fit(X)
print model

In [ ]:
Counter(model.labels_)

### Plot the cluster in feature space

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sctr = ax.scatter(X[:, 0], X[:, 1], c=model.labels_, s=140, alpha=0.9,
                 cmap=plt.cm.Set1)
fig.show()

# DBSCAN with GraphLab

In [ ]:
import graphlab as gl

sf = gl.SFrame(X)
sf = sf.unpack('X1')
model2 = gl.dbscan.create(sf, radius=0.25, min_core_neighbors=10)

In [ ]:
model2.summary()

### Access cluster labels and instance types

In [ ]:
model2['cluster_id']

### Plot by instances

In [ ]:
sf_out = sf.add_row_number('row_id')
sf_out = sf_out.join(model2['cluster_id'], on='row_id', how='left')

In [ ]:
type_map = {k: v for v, k in enumerate(sf_out['type'].unique())}
sf_out['type_code'] = sf_out['type'].apply(lambda x: type_map[x])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(sf_out['X1.0'], sf_out['X1.1'], c=sf_out['type_code'],
           s=140, alpha=0.9, cmap=plt.cm.Set1)
fig.show()

# Document Deduplication with DBSCAN

### download dataset

In [ ]:
import os
if os.path.exists('wikipedia_w0'):
    wiki = gl.SFrame('wikipedia_w0')
else:
    wiki = gl.SFrame.read_csv('https://static.turi.com/datasets/wikipedia/raw/w0', header=False)
    wiki.save('wikipedia_w0')

In [ ]:
#wiki = gl.SFrame('wikipedia_w0')
wiki = wiki.add_row_number('row_id')
wiki.print_rows(5, max_column_width=70, max_row_width=90)

### Feature Engineering

### Convert to bag-of-words

In [ ]:
wiki['words'] = gl.text_analytics.count_words(wiki['X1'])
wiki[['words']].print_rows(5, max_column_width=80)

### Remove stop-words

In [ ]:
wiki['words'] = wiki['words'].dict_trim_by_keys(gl.text_analytics.stopwords(),
                                                exclude=True)

### Create Model

In [ ]:
# dbscan = gl.dbscan.create(wiki, features=['words'],
#                          distance='jaccard', radius=0.6,
#                          min_core_neighbors=3, verbose=False)

# dbscan.save('wiki_dbscan.gl')
dbscan = gl.load_model('wiki_dbscan.gl')
dbscan.summary()

In [ ]:
dbscan['cluster_id'].show()

### Explore the cluster

In [ ]:
wiki = wiki.join(dbscan['cluster_id'], on='row_id', how='left')

In [ ]:
wiki[wiki['cluster_id'] == 50][['X1']].print_rows(5, max_row_width=80,
                                                  max_column_width=80)